# Transforming the ethnicity dataset

This notebook transforms the cleaned ethnicity dataset `ibe040-migration-by-ethnicity-2016-2025-interim.csv` into an analysis-ready wide format dataset. The goal is to pivot `Rādītāji` column into two, add a new column named `Migrācijas saldo` (`Iebrauca Latvijā - Izbrauca no Latvijas`) and save the dataset in `data/processed` for further work.

In [1]:
import pandas as pd

ethnicity_data = pd.read_csv("../data/interim/ibe040-migration-by-ethnicity-2016-2025-interim.csv")

ethnicity_data.head()

,Tautība,Rādītāji,Gads,Skaits
0,Pavisam,Iebrauca Latvijā,2016,8345.0
1,Pavisam,Izbrauca no Latvijas,2016,20574.0
2,Latvieši,Iebrauca Latvijā,2016,3081.0
3,Latvieši,Izbrauca no Latvijas,2016,10379.0
4,Afgāņi,Iebrauca Latvijā,2016,NaN


The required libraries are imported and the cleaned ethnicity dataset is loaded.
The first rows confirm that the dataset was loaded as intended from the cleaning work, with all expected columns and values.

In [2]:
ethnicity_pivoted = ethnicity_data.pivot(
    index=["Tautība", "Gads"],
    columns="Rādītāji",
    values="Skaits",
).reset_index()

ethnicity_pivoted.columns.name = None

ethnicity_pivoted.head()

,Tautība,Gads,Iebrauca Latvijā,Izbrauca no Latvijas
0,Afgāņi,2016,NaN,NaN
1,Afgāņi,2017,NaN,NaN
2,Afgāņi,2018,NaN,NaN
3,Afgāņi,2019,NaN,NaN
4,Afgāņi,2020,NaN,NaN


The dataset is transformed from long to wide format using `.pivot()`. This method reshapes the dataset without aggregating the migration counts and preserves missing values. Unlike `.pivot_table()`, `.pivot()` requires each combination of identifier columns and migration indicator to be unique. If duplicate combinations appear in future data, `.pivot()` raises an error rather than silently aggregating them. Additionally, unavailable migration counts should remain missing rather than being aggregated into another value, to avoid compromising the data.

`Tautība` and `Gads` are used as identifier columns, so each row represents one ethnicity in one year. The values in `Rādītāji` column are converted into two separate columns that were the values of the column: `Iebrauca Latvijā` and `Izbrauca no Latvijas`. The migration counts from `Skaits` column are placed into these new columns.

`.reset_index()` moves `Tautība` and `Gads` back into normal columns after pivoting.

`ethnicity_pivoted.columns.name = None` removes `Rādītāji` column-axis name that pandas keeps above the new columns after pivoting. It does not remove the columns themselves or change the data values; it only removes the unnecessary column-axis label.

The displayed rows confirm that the dataset pivot was executed successfully. The rows also show that missing migration counts remain missing, with the value `NaN`.

In [3]:
ethnicity_pivoted.shape

(480, 4)

The pivoted dataset contains 480 rows and 4 columns.
The rows represent every combination of 48 ethnicity categories and ten years.
The four columns now are `Tautība`, `Gads`, `Iebrauca Latvijā`, and `Izbrauca no Latvijas`.

In [4]:
ethnicity_pivoted["Migrācijas saldo"] = (
    ethnicity_pivoted["Iebrauca Latvijā"] - ethnicity_pivoted["Izbrauca no Latvijas"])

ethnicity_pivoted.head()

,Tautība,Gads,Iebrauca Latvijā,Izbrauca no Latvijas,Migrācijas saldo
0,Afgāņi,2016,NaN,NaN,NaN
1,Afgāņi,2017,NaN,NaN,NaN
2,Afgāņi,2018,NaN,NaN,NaN
3,Afgāņi,2019,NaN,NaN,NaN
4,Afgāņi,2020,NaN,NaN,NaN


A new column, `Migrācijas saldo`, is calculated as immigration minus emigration: `Migrācijas saldo = Iebrauca Latvijā - Izbrauca no Latvijas`. A negative migration saldo means that more people left Latvia than arrived. A positive migration saldo means that more people arrived in Latvia than left.

Pandas propagates missing values through subtraction, therefore migration saldo remains missing where either immigration or emigration is missing. Missing values are not treated as zero.

The dataset confirms that migration saldo remains missing for the rows where one or the other, or both of the values in the split `Rādītāji` column are unavailable.

In [5]:
ethnicity_pivoted.to_csv(
    "../data/processed/ibe040-migration-by-ethnicity-2016-2025-processed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("The dataset has been saved")

The dataset has been saved


The processed wide format dataset is saved to the `data/processed` folder for use in analysis as the next step in the planned methodology.
`index=False` prevents pandas from writing its numeric row index as an extra column in the `.csv` file. `encoding="utf-8-sig"` writes a UTF-8 file with a byte order mark, which helps some programs, including Microsoft Excel, to correctly recognize the encoding and display Latvian characters.